# Week 1 — First Machine Learning Workflow and commit test

**Machine Learning for Engineering Applications**

This notebook is deliberately simple. The aim is not to build the world's best model. It is to learn to ask:

> **It runs — but is it credible?**

### By the end of this notebook you should be able to:
- load and inspect an engineering dataset;
- choose input features and a target;
- see exactly which observations are placed in training and test sets;
- compare a simple baseline with a machine-learning model;
- change one feature and observe what happens;
- compare a random row split with a split by machine;
- identify at least one reason why an apparently excellent result may be invalid.

The dataset is **synthetic** and is designed for teaching rather than representing a real machine.


## 1. Imports

If this cell fails, make sure your environment contains:

`pandas`, `numpy`, `matplotlib`, `scikit-learn`


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

RANDOM_STATE = 7


Matplotlib is building the font cache; this may take a moment.


## 2. Load the dataset

The notebook expects this repository structure:

```text
.
├── week01_demo.ipynb
└── data
    └── week01_demo.csv
```


In [2]:
df = pd.read_csv("data/week01_demo.csv")

print(f"Rows: {len(df):,}")
print(f"Machines: {df['machine_id'].nunique()}")
df.head()


Rows: 660
Machines: 12


,machine_id,reading_no,hours_since_start,load,temperature,vibration_rms,shaft_speed_rpm,post_inspection_RUL_estimate_hours,remaining_life_hours
0,M01,0,0.000,56.431,35.620,1.1854,900.70,305.341,305.275
1,M01,1,4.700,58.254,34.822,1.1463,903.14,303.676,302.797
2,M01,2,9.401,57.786,35.544,1.2460,907.03,295.961,296.389
3,M01,3,14.101,60.023,36.805,1.1804,917.77,293.434,291.293
4,M01,4,18.801,58.685,37.144,1.1952,910.60,286.454,286.568


## 3. What is in the data?

Each row is one measurement from a machine at a particular point in its operating history.

The prediction target is:

**`remaining_life_hours`**

Candidate inputs include:

- `load`
- `temperature`
- `vibration_rms`
- `shaft_speed_rpm`

There is also one intentionally problematic column:

**`post_inspection_RUL_estimate_hours`**

Do not use it initially. Ask yourself *when* that measurement would actually become available.


In [ ]:
df.describe(include="all").T


## 4. Plot one machine before modelling

Before fitting a model, look at the data.

What happens to vibration as the machine gets closer to the end of its life?


In [ ]:
machine_to_plot = "M01"
one_machine = df[df["machine_id"] == machine_to_plot]

plt.figure(figsize=(8, 4))
plt.scatter(
    one_machine["remaining_life_hours"],
    one_machine["vibration_rms"],
    alpha=0.8
)
plt.xlabel("Remaining life [hours]")
plt.ylabel("Vibration RMS")
plt.title(f"{machine_to_plot}: vibration vs remaining life")
plt.gca().invert_xaxis()
plt.grid(alpha=0.25)
plt.show()


## 5. Student controls — change these

This is the main cell to edit during the Week 1 activity.

### Try changing:
1. **`FEATURES`** — remove or add one input.
2. **`SPLIT_MODE`** — switch from `"random"` to `"by_machine"`.

Start with the three features shown on the lecture slide.


In [ ]:
# ============================================================
# STUDENT CONTROLS — CHANGE THESE AND RE-RUN FROM HERE DOWN
# ============================================================

FEATURES = [
    "load",
    "temperature",
    "vibration_rms",
]

TARGET = "remaining_life_hours"

# Choose either:
#   "random"      -> individual rows are randomly divided
#   "by_machine"  -> whole machines are held out for testing
SPLIT_MODE = "random"

TEST_SIZE = 0.25

print("Features:", FEATURES)
print("Target:", TARGET)
print("Split mode:", SPLIT_MODE)


## 6. Create the train/test split

This cell deliberately makes the split visible.

### Question
If several measurements come from the **same physical machine**, should measurements from that machine appear in both training and testing?


In [ ]:
X = df[FEATURES]
y = df[TARGET]

if SPLIT_MODE == "random":
    train_idx, test_idx = train_test_split(
        np.arange(len(df)),
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )

elif SPLIT_MODE == "by_machine":
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )
    train_idx, test_idx = next(
        splitter.split(X, y, groups=df["machine_id"])
    )

else:
    raise ValueError("SPLIT_MODE must be 'random' or 'by_machine'.")

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_meta = df.iloc[train_idx][["machine_id", "reading_no"]]
test_meta = df.iloc[test_idx][["machine_id", "reading_no"]]

print(f"Training rows: {len(train_idx)}")
print(f"Test rows:     {len(test_idx)}")


## 7. See exactly how the data were split

Look at the machine IDs below.

With a **random row split**, the same machine will usually appear in both sets.

With a **by-machine split**, the test machines should be completely unseen during training.


In [ ]:
train_machines = sorted(train_meta["machine_id"].unique())
test_machines = sorted(test_meta["machine_id"].unique())
overlap = sorted(set(train_machines).intersection(test_machines))

print("Machines in TRAIN:")
print(train_machines)

print("\nMachines in TEST:")
print(test_machines)

print("\nMachines appearing in BOTH:")
print(overlap if overlap else "None")


In [ ]:
split_view = df[["machine_id", "reading_no", "remaining_life_hours"]].copy()
split_view["set"] = "not used"
split_view.loc[train_idx, "set"] = "train"
split_view.loc[test_idx, "set"] = "test"

split_view.sort_values(["machine_id", "reading_no"]).head(20)


## 8. Baseline model

A baseline answers:

> **Can the ML model beat something embarrassingly simple?**

For regression, we will start by predicting the **mean training-set remaining life** for every test observation.


In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_r2 = r2_score(y_test, baseline_pred)

print(f"Baseline MAE: {baseline_mae:.2f} hours")
print(f"Baseline R²:  {baseline_r2:.3f}")


## 9. A first machine-learning model

We use a Random Forest because it can capture nonlinear relationships with very little setup.

At this stage, focus on the **workflow and experimental design**, not on tuning the algorithm.


In [ ]:
model = RandomForestRegressor(
    n_estimators=250,
    min_samples_leaf=2,
    random_state=RANDOM_STATE
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

model_mae = mean_absolute_error(y_test, pred)
model_r2 = r2_score(y_test, pred)

print(f"Random Forest MAE: {model_mae:.2f} hours")
print(f"Random Forest R²:  {model_r2:.3f}")


## 10. Compare predictions with reality

A numerical score is useful, but always look at the predictions too.


In [ ]:
plt.figure(figsize=(5.5, 5.5))
plt.scatter(y_test, pred, alpha=0.65)
limits = [
    min(float(y_test.min()), float(pred.min())),
    max(float(y_test.max()), float(pred.max()))
]
plt.plot(limits, limits, "--", linewidth=1)
plt.xlim(limits)
plt.ylim(limits)
plt.xlabel("Actual remaining life [hours]")
plt.ylabel("Predicted remaining life [hours]")
plt.title(f"Predicted vs actual — {SPLIT_MODE} split")
plt.grid(alpha=0.25)
plt.show()


## 11. Which features did the model use?

Feature importance is **not** proof of causality, but it can help us ask better engineering questions.


In [ ]:
importance = (
    pd.Series(model.feature_importances_, index=FEATURES)
    .sort_values(ascending=False)
)

display(importance.to_frame("importance"))

plt.figure(figsize=(7, 3.5))
importance.sort_values().plot(kind="barh")
plt.xlabel("Random Forest feature importance")
plt.title("Which inputs influenced the model?")
plt.tight_layout()
plt.show()


# In-class experiment A — change one feature

Go back to the **STUDENT CONTROLS** cell.

Try at least three cases:

```python
FEATURES = ["load"]
```

```python
FEATURES = ["vibration_rms"]
```

```python
FEATURES = ["load", "temperature", "vibration_rms", "shaft_speed_rpm"]
```

For each case, record:
- MAE;
- R²;
- whether the result makes engineering sense.

### Discussion prompt
A feature improving the score does **not automatically** mean it is a good feature. Why?


# In-class experiment B — change the split

Now keep the same features and change:

```python
SPLIT_MODE = "random"
```

to:

```python
SPLIT_MODE = "by_machine"
```

Then rerun from the controls cell downward.

### Look carefully at:
- which machines are in training;
- which machines are in testing;
- whether there is any overlap;
- how the MAE and R² change.

### Engineering question
Which split better represents the claim:

> “My model can predict remaining life on a machine it has never seen before”?


# In-class experiment C — the suspiciously good model

Only after discussing it with your neighbour, try adding:

```python
"post_inspection_RUL_estimate_hours"
```

to `FEATURES`.

You may obtain an extremely impressive result.

### Before celebrating, answer:
1. When is this measurement taken?
2. Would it exist at the time a live prediction must be made?
3. Is the model learning the engineering problem we claim it is solving?

This is a deliberately constructed example of **data leakage**.


## 12. Optional comparison cell

This cell runs the same model with both split strategies so you can compare them directly using the current `FEATURES`.


In [ ]:
def evaluate_split(split_mode, features):
    X_local = df[features]
    y_local = df[TARGET]

    if split_mode == "random":
        tr, te = train_test_split(
            np.arange(len(df)),
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE
        )
    else:
        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=TEST_SIZE,
            random_state=RANDOM_STATE
        )
        tr, te = next(
            splitter.split(X_local, y_local, groups=df["machine_id"])
        )

    m = RandomForestRegressor(
        n_estimators=250,
        min_samples_leaf=2,
        random_state=RANDOM_STATE
    )
    m.fit(X_local.iloc[tr], y_local.iloc[tr])
    p = m.predict(X_local.iloc[te])

    return {
        "split": split_mode,
        "features": ", ".join(features),
        "MAE_hours": mean_absolute_error(y_local.iloc[te], p),
        "R2": r2_score(y_local.iloc[te], p),
        "train_machines": df.iloc[tr]["machine_id"].nunique(),
        "test_machines": df.iloc[te]["machine_id"].nunique(),
        "machine_overlap": len(
            set(df.iloc[tr]["machine_id"]).intersection(
                set(df.iloc[te]["machine_id"])
            )
        ),
    }

comparison = pd.DataFrame([
    evaluate_split("random", FEATURES),
    evaluate_split("by_machine", FEATURES),
])

comparison


# Final Week 1 reflection

Write a short note in your own words:

### 1. What changed when you changed one feature?

### 2. What changed when you split by machine rather than by row?

### 3. What would make this ML result credible enough to show to an engineering manager?

A strong answer might discuss:
- whether the test data are genuinely unseen;
- whether features are available at prediction time;
- whether the model beats a simple baseline;
- whether the test case represents the intended deployment case;
- whether the result makes physical sense.

**Commit your answer to Git with the notebook.**
